# 03 - Sequence preparation (inspection)
Mirrors what `scripts/Model_Training.py` does: the per-building split policy (`scripts/splitting.py`: full-history training for buildings with about a year of data, a chronological holdout when there is enough), `MinMaxScaler` fitted on the **training part only**,
24-step sliding windows over `FEATURE_COLS`. Use it to inspect the arrays; the training script does not read these files.

Reads `data/processed/<building>.csv` (written by `scripts/Model_Training.py` or `scripts/pipeline.py`).
Writes `data/sequences/<building>.npz`. Scalers are written to `data/scalers/` so the production scalers in `scalers/` are not overwritten.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
for p in (PROJECT_ROOT, PROJECT_ROOT / "scripts"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from src.config import BUILDING_SPLIT, BUILDINGS, CONFIG, FEATURE_COLS

In [ ]:
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

from scripts.scaling import scale_features
from scripts.splitting import decide_split
from scripts.sequence_creations import create_sequences

processed_path = PROJECT_ROOT / "data" / "processed"
sequence_path = PROJECT_ROOT / "data" / "sequences"
scaler_path = PROJECT_ROOT / "data" / "scalers"
sequence_path.mkdir(parents=True, exist_ok=True)
scaler_path.mkdir(parents=True, exist_ok=True)

WINDOW_SIZE = int(CONFIG["window_size"])

### Main processing loop (only the configured buildings)

In [ ]:
arrays = {}
for building in BUILDINGS:
    file = processed_path / f"{building}.csv"
    if not file.is_file():
        print("Missing, skipped:", file)
        continue

    df = pd.read_csv(file, parse_dates=["timestamp"]).sort_values("timestamp").reset_index(drop=True)

    decision = decide_split(pd.DatetimeIndex(df["timestamp"]), building, CONFIG, BUILDING_SPLIT)
    print(decision["message"])
    X_train_sc, X_test_sc, scaler, split_idx = scale_features(df, FEATURE_COLS, split_idx=decision["split_idx"])
    joblib.dump(scaler, scaler_path / f"{building}_scaler.pkl")

    X_train = create_sequences(X_train_sc, WINDOW_SIZE)
    X_test = create_sequences(X_test_sc, WINDOW_SIZE)
    np.savez(sequence_path / f"{building}.npz", X_train=X_train, X_test=X_test, window_size=WINDOW_SIZE)

    arrays[building] = X_train
    print(building, "train", X_train.shape, "test", X_test.shape)

### Visual checks

In [ ]:
building = next(iter(arrays))
X_train = arrays[building]

fig, axes = plt.subplots(1, 3, figsize=(16, 3.5))
axes[0].plot(X_train[100][:, 0]); axes[0].set_title(f"{building}: energy across one {WINDOW_SIZE}h window")
axes[1].plot(X_train[0][:, 0], label="window 0"); axes[1].plot(X_train[1][:, 0], label="window 1")
axes[1].set_title("Consecutive windows (shifted by one step)"); axes[1].legend()
axes[2].hist(X_train[:, :, 0].ravel(), bins=50); axes[2].set_title("Scaled energy distribution (0..1)")
plt.tight_layout(); plt.show()

In [ ]:
# Integrity checks for every building
for building, X in arrays.items():
    assert X.ndim == 3 and X.shape[1] == WINDOW_SIZE and X.shape[2] == len(FEATURE_COLS), building
    assert not np.isnan(X).any(), building
    print(f"{building}: windows={len(X)}  min={X.min():.3f}  max={X.max():.3f}  "
          f"lowest-variance feature={FEATURE_COLS[int(np.var(X, axis=(0, 1)).argmin())]}")